In [1]:
from google.colab import files
uploaded = files.upload()

Saving tes.json to tes (2).json


In [2]:
import time
import gc
import string
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)

# ==========================================
# CONFIGURATIONS & HYPERPARAMETERS
# ==========================================
SEED = 42
MAX_LENGTH = 384
DOC_STRIDE = 128
N_BEST = 20
MAX_ANSWER_LENGTH = 30

# Training Hyperparameters
BATCH_SIZE = 8
LEARNING_RATE = 3e-5
EPOCHS = 3
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

OUTPUT_DIR = "./results"

# Definisikan model-model yang ingin dibandingkan
MODEL_CONFIGS = [
    {
        "model_name": "BERT-QA",
        "checkpoint": "bert-base-uncased",
        "output_name": "bert_qa"
    }
]

In [3]:

import json
import shutil
import os
import pandas as pd


# Karena diunggah menggunakan files.upload(), filenya langsung berada di root folder Colab
file_name = "tes.json"

# Load JSON
with open(file_name, "r", encoding="utf-8") as f:
    data = json.load(f)

samples = []

# Memastikan kita mengambil list data utama, baik formatnya langsung list atau dibungkus key tertentu
if isinstance(data, dict) and "data" in data:
    items = data["data"]
elif isinstance(data, list):
    items = data
else:
    items = next((v for v in data.values() if isinstance(v, list)), [])

# Looping ekstrak QA dataset sesuai format barumu
for item in items:
    question = item.get("question", "")
    context = item.get("context", "")

    # Ambil dictionary answers
    answers_dict = item.get("answers", {})

    # Ambil list di dalam key "text"
    text_list = answers_dict.get("text", [])

    # Ekstrak teks jawaban pertama jika list-nya tersedia
    if len(text_list) > 0:
        answer = text_list[0]
    else:
        answer = None

    samples.append({
        "question": question,
        "context": context,
        "answer": answer
    })

# Convert ke DataFrame
df = pd.DataFrame(samples)

print("\nHasil 5 data pertama:")
print(df.head())


Hasil 5 data pertama:
                                            question  \
0  Berapa rasio anak perempuan di Indonesia yang ...   
1  Berapa persentase tingkat mutilasi alat kelami...   
2     Anak laki-laki di sekolah rentan terhadap apa?   
3  Berapa anggaran pemerintah untuk perlindungan ...   
4  Berapa persen anak di bawah usia 18 tahun yang...   

                                             context                  answer  
0  Satu dari sembilan anak perempuan di Indonesia...      Satu dari sembilan  
1  Tingkat mutilasi alat kelamin perempuan di Ind...               52 persen  
2  Anak laki-laki di sekolah rentan terhadap sera...          serangan fisik  
3  Anggaran pemerintah untuk perlindungan anak ku...  kurang dari 0,1 persen  
4  Sekitar 17 persen anak di bawah usia 18 tahun ...               17 persen  


In [4]:
# Pastikan library ter-install
# !pip install transformers>=4.48.0 datasets evaluate accelerate scikit-learn torch pandas numpy

import json
import time
import gc
import string
import os
import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import GroupShuffleSplit
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)

# ==========================================
# CONFIGURATIONS & HYPERPARAMETERS
# ==========================================
SEED = 42
MAX_LENGTH = 384
DOC_STRIDE = 128
N_BEST = 20
MAX_ANSWER_LENGTH = 30

# Training Hyperparameters
BATCH_SIZE = 8
LEARNING_RATE = 3e-5
EPOCHS = 3
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

OUTPUT_DIR = "./results"
file_name = "tes.json"

# Definisikan model-model yang ingin dibandingkan sesuai permintaan
MODEL_CONFIGS = [
    {
        "model_name": "BERT-base-uncased",
        "checkpoint": "bert-base-uncased",
        "output_name": "bert_base_uncased_qa"
    },
    {
        "model_name": "ModernBERT-base",
        "checkpoint": "answerdotai/ModernBERT-base",
        "output_name": "modernbert_base_qa"
    }
]

# ==========================================
# 1. LOAD DAN VALIDASI DATASET
# ==========================================

def load_custom_qa_data(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    if isinstance(data, dict) and "data" in data:
        items = data["data"]
    elif isinstance(data, list):
        items = data
    else:
        raise ValueError("Format JSON tidak dikenali. Gunakan list atau dict dengan key 'data'.")

    records = []
    corrected_offsets = 0
    corrected_case = 0
    unresolved = []

    for index, item in enumerate(items):
        article_id = item.get("article_id")
        context = item.get("context", "")
        question = item.get("question", "")
        answers = item.get("answers", {})

        answer_texts = answers.get("text", [])
        answer_starts = answers.get("answer_start", [])

        if not article_id or not context or not question:
            unresolved.append((index, article_id, "Missing article_id/context/question"))
            continue

        if not answer_texts or not answer_starts:
            unresolved.append((index, article_id, "Missing answer"))
            continue

        answer_text = answer_texts[0]
        answer_start = answer_starts[0]

        # Offset sudah benar.
        if context[answer_start:answer_start + len(answer_text)] == answer_text:
            pass
        else:
            # Cari kecocokan persis di seluruh context.
            occurrences = []
            position = 0
            while True:
                position = context.find(answer_text, position)
                if position == -1:
                    break
                occurrences.append(position)
                position += 1

            if len(occurrences) == 1:
                answer_start = occurrences[0]
                corrected_offsets += 1

            elif len(occurrences) > 1:
                nearest = min(occurrences, key=lambda x: abs(x - answer_start))
                if abs(nearest - answer_start) <= 5:
                    answer_start = nearest
                    corrected_offsets += 1
                else:
                    unresolved.append((index, article_id, "Ambiguous answer position"))
                    continue

            else:
                # Perbaiki perbedaan kapitalisasi hanya jika ada satu kecocokan
                # case-insensitive dengan panjang karakter yang sama.
                lower_context = context.lower()
                lower_answer = answer_text.lower()
                lower_occurrences = []
                position = 0

                while True:
                    position = lower_context.find(lower_answer, position)
                    if position == -1:
                        break
                    lower_occurrences.append(position)
                    position += 1

                if len(lower_occurrences) == 1:
                    answer_start = lower_occurrences[0]
                    answer_text = context[answer_start:answer_start + len(answer_text)]
                    corrected_case += 1
                else:
                    unresolved.append((index, article_id, "Answer not found or ambiguous in context"))
                    continue

        # Validasi akhir: jawaban harus benar-benar merupakan substring context.
        if context[answer_start:answer_start + len(answer_text)] != answer_text:
            unresolved.append((index, article_id, "Final answer-span validation failed"))
            continue

        records.append({
            "article_id": str(article_id),
            "context": context,
            "question": question,
            "answers": {
                "text": [answer_text],
                "answer_start": [int(answer_start)],
            },
        })

    print("Original samples       :", len(items))
    print("Valid samples          :", len(records))
    print("Corrected offsets      :", corrected_offsets)
    print("Corrected capitalization:", corrected_case)
    print("Unresolved samples     :", len(unresolved))

    if unresolved:
        print("\nSamples requiring manual inspection:")
        for sample in unresolved:
            print(sample)

    if not records:
        raise ValueError("Tidak ada sampel QA yang valid.")

    return Dataset.from_list(records)

hf_dataset = load_custom_qa_data(file_name)

# -----------------------------
# 3. ARTICLE-LEVEL TRAIN/VAL/TEST SPLIT
# -----------------------------

def create_article_level_split(dataset):
    groups = np.asarray(dataset["article_id"])
    indices = np.arange(len(dataset))
    target = np.array([0.70, 0.15, 0.15])

    best_split = None

    # Mencari pembagian grup yang mendekati proporsi 70:15:15.
    for seed in range(500):
        train_val_idx, test_idx = next(
            GroupShuffleSplit(
                n_splits=1,
                test_size=0.15,
                random_state=seed,
            ).split(indices, groups=groups)
        )

        train_rel, val_rel = next(
            GroupShuffleSplit(
                n_splits=1,
                test_size=0.15 / 0.85,
                random_state=seed + 10000,
            ).split(
                train_val_idx,
                groups=groups[train_val_idx],
            )
        )

        train_idx = train_val_idx[train_rel]
        val_idx = train_val_idx[val_rel]

        proportions = np.array([
            len(train_idx),
            len(val_idx),
            len(test_idx),
        ]) / len(dataset)

        score = np.sum((proportions - target) ** 2)

        if best_split is None or score < best_split[0]:
            best_split = (score, train_idx, val_idx, test_idx, seed)

    _, train_idx, val_idx, test_idx, selected_seed = best_split

    split_data = DatasetDict({
        "train": dataset.select(train_idx.tolist()),
        "validation": dataset.select(val_idx.tolist()),
        "test": dataset.select(test_idx.tolist()),
    })

    train_articles = set(split_data["train"]["article_id"])
    val_articles = set(split_data["validation"]["article_id"])
    test_articles = set(split_data["test"]["article_id"])

    assert train_articles.isdisjoint(val_articles)
    assert train_articles.isdisjoint(test_articles)
    assert val_articles.isdisjoint(test_articles)

    print("\n--- DATASET SPLIT SUMMARY ---")
    for split_name, split in split_data.items():
        print(
            f"{split_name}: {len(split)} samples | "
            f"{len(set(split['article_id']))} articles"
        )

    print("Article overlap: None")
    print("Selected split seed:", selected_seed)

    return split_data

dataset_dict = create_article_level_split(hf_dataset)

# -----------------------------
# 4. TOKENIZATION DAN ALIGNMENT ANSWER
# -----------------------------

def prepare_qa_features(examples, tokenizer):
    questions = [question.lstrip() for question in examples["question"]]

    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized["offset_mapping"]

    tokenized["start_positions"] = []
    tokenized["end_positions"] = []
    tokenized["example_id"] = []
    tokenized["offset_mapping"] = []

    for feature_index, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][feature_index]

        # Penanganan fallback aman untuk CLS / BOS token
        cls_token_id = tokenizer.cls_token_id if tokenizer.cls_token_id is not None else tokenizer.bos_token_id
        cls_index = input_ids.index(cls_token_id) if (cls_token_id is not None and cls_token_id in input_ids) else 0

        sequence_ids = tokenized.sequence_ids(feature_index)

        sample_index = sample_mapping[feature_index]
        answers = examples["answers"][sample_index]
        context = examples["context"][sample_index]

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        # Hanya offset token pada bagian context yang dipertahankan
        context_offsets = [
            list(offset) if (sequence_ids[token_index] == 1 and offset is not None) else None
            for token_index, offset in enumerate(offsets)
        ]

        token_start_index = 0
        while (
            token_start_index < len(sequence_ids)
            and sequence_ids[token_start_index] != 1
        ):
            token_start_index += 1

        token_end_index = len(sequence_ids) - 1
        while token_end_index >= 0 and sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        start_position = cls_index
        end_position = cls_index

        if token_start_index <= token_end_index:
            if (
                offsets[token_start_index][0] <= start_char
                and offsets[token_end_index][1] >= end_char
            ):
                while (
                    token_start_index <= token_end_index
                    and offsets[token_start_index][0] <= start_char
                ):
                    token_start_index += 1

                start_position = token_start_index - 1

                while (
                    token_end_index >= 0
                    and offsets[token_end_index][1] >= end_char
                ):
                    token_end_index -= 1

                end_position = token_end_index + 1

        tokenized["start_positions"].append(start_position)
        tokenized["end_positions"].append(end_position)
        tokenized["example_id"].append(sample_index)
        tokenized["offset_mapping"].append(context_offsets)

    return tokenized

# -----------------------------
# 5. TEXT-LEVEL EXACT MATCH DAN TOKEN F1
# -----------------------------

def normalize_answer(text):
    text = text.lower()
    text = "".join(char for char in text if char not in string.punctuation)
    return " ".join(text.split())

def calculate_text_f1(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    true_tokens = normalize_answer(ground_truth).split()

    if not pred_tokens or not true_tokens:
        return float(pred_tokens == true_tokens)

    common = {}
    for token in pred_tokens:
        common[token] = common.get(token, 0) + 1

    overlap = 0
    true_counts = {}
    for token in true_tokens:
        true_counts[token] = true_counts.get(token, 0) + 1

    for token, count in common.items():
        overlap += min(count, true_counts.get(token, 0))

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(true_tokens)
    return 2 * precision * recall / (precision + recall)

def compute_text_qa_metrics(eval_pred, eval_features, raw_examples):
    start_logits, end_logits = eval_pred.predictions

    best_predictions = {}

    for feature_index, feature in enumerate(eval_features):
        example_id = int(feature["example_id"])
        offsets = feature["offset_mapping"]
        context = raw_examples[example_id]["context"]

        start_indexes = np.argsort(start_logits[feature_index])[-N_BEST:][::-1]
        end_indexes = np.argsort(end_logits[feature_index])[-N_BEST:][::-1]

        for start_index in start_indexes:
            if start_index >= len(offsets) or offsets[start_index] is None:
                continue

            for end_index in end_indexes:
                if end_index < start_index or end_index >= len(offsets):
                    continue
                if offsets[end_index] is None:
                    continue
                if end_index - start_index + 1 > MAX_ANSWER_LENGTH:
                    continue

                start_char = offsets[start_index][0]
                end_char = offsets[end_index][1]

                if end_char <= start_char:
                    continue

                prediction_text = context[start_char:end_char]
                score = (
                    float(start_logits[feature_index][start_index])
                    + float(end_logits[feature_index][end_index])
                )

                current = best_predictions.get(example_id)
                if current is None or score > current["score"]:
                    best_predictions[example_id] = {
                        "text": prediction_text,
                        "score": score,
                    }

    exact_matches = []
    f1_scores = []

    for example_id, example in enumerate(raw_examples):
        prediction = best_predictions.get(example_id, {}).get("text", "")
        gold_answers = example["answers"]["text"]

        gold = gold_answers[0] if gold_answers else ""

        exact_matches.append(
            float(normalize_answer(prediction) == normalize_answer(gold))
        )
        f1_scores.append(calculate_text_f1(prediction, gold))

    return {
        "exact_match": 100.0 * float(np.mean(exact_matches)),
        "f1": 100.0 * float(np.mean(f1_scores)),
    }

# -----------------------------
# 6. TRAINING DAN EVALUASI MODEL
# -----------------------------

def train_and_evaluate_qa(model_config, raw_splits):
    model_name = model_config["model_name"]
    checkpoint = model_config["checkpoint"]
    output_name = model_config["output_name"]

    print(f"\n{'=' * 55}")
    print(f"Memulai eksperimen: {model_name}")
    print(f"{'=' * 55}")

    experiment_start = time.time()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    gc.collect()

    tokenizer = AutoTokenizer.from_pretrained(checkpoint)

    print("Tokenisasi train...")
    train_features = raw_splits["train"].map(
        lambda batch: prepare_qa_features(batch, tokenizer),
        batched=True,
        remove_columns=raw_splits["train"].column_names,
        desc=f"Tokenizing {model_name} train",
    )

    print("Tokenisasi validation...")
    validation_features = raw_splits["validation"].map(
        lambda batch: prepare_qa_features(batch, tokenizer),
        batched=True,
        remove_columns=raw_splits["validation"].column_names,
        desc=f"Tokenizing {model_name} validation",
    )

    print("Tokenisasi test...")
    test_features = raw_splits["test"].map(
        lambda batch: prepare_qa_features(batch, tokenizer),
        batched=True,
        remove_columns=raw_splits["test"].column_names,
        desc=f"Tokenizing {model_name} test",
    )

    validation_metadata = [
        {
            "example_id": int(validation_features[i]["example_id"]),
            "offset_mapping": validation_features[i]["offset_mapping"],
        }
        for i in range(len(validation_features))
    ]
    test_metadata = [
        {
            "example_id": int(test_features[i]["example_id"]),
            "offset_mapping": test_features[i]["offset_mapping"],
        }
        for i in range(len(test_features))
    ]

    train_model_dataset = train_features.remove_columns(
        ["example_id", "offset_mapping"]
    )
    validation_model_dataset = validation_features.remove_columns(
        ["example_id", "offset_mapping"]
    )
    test_model_dataset = test_features.remove_columns(
        ["example_id", "offset_mapping"]
    )

    def validation_metrics(eval_pred):
        return compute_text_qa_metrics(
            eval_pred,
            validation_metadata,
            raw_splits["validation"],
        )

    model = AutoModelForQuestionAnswering.from_pretrained(checkpoint)

    model_output_dir = os.path.join(OUTPUT_DIR, output_name)

    # Menjamin kompatibilitas mutlak lintas berbagai versi Transformers dengan penanganan try-except dinamis
    try:
        training_args = TrainingArguments(
            output_dir=model_output_dir,
            evaluation_strategy="epoch",
            save_strategy="epoch",
            learning_rate=LEARNING_RATE,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            num_train_epochs=EPOCHS,
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True,
            report_to="none",
            seed=SEED,
            data_seed=SEED,
            fp16=torch.cuda.is_available(),
            save_total_limit=1,
        )
    except TypeError:
        training_args = TrainingArguments(
            output_dir=model_output_dir,
            eval_strategy="epoch",
            save_strategy="epoch",
            learning_rate=LEARNING_RATE,

            weight_decay=WEIGHT_DECAY,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            num_train_epochs=EPOCHS,
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True,
            report_to="none",
            seed=SEED,
            data_seed=SEED,
            fp16=torch.cuda.is_available(),
            save_total_limit=1,
        )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_model_dataset,
        eval_dataset=validation_model_dataset,
        compute_metrics=validation_metrics,

        data_collator=DefaultDataCollator(),
    )

    print(f"Training {model_name}...")
    train_result = trainer.train()

    if torch.cuda.is_available():
        train_peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
        torch.cuda.reset_peak_memory_stats()
    else:
        train_peak_vram_mb = 0.0

    def test_metrics(eval_pred):
        return compute_text_qa_metrics(
            eval_pred,
            test_metadata,
            raw_splits["test"],
        )

    trainer.compute_metrics = test_metrics

    print(f"Evaluasi final pada test set: {model_name}...")
    inference_start = time.time()
    test_results = trainer.evaluate(
        eval_dataset=test_model_dataset,
        metric_key_prefix="test",
    )
    measured_inference_runtime = time.time() - inference_start

    if torch.cuda.is_available():
        test_peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        test_peak_vram_mb = 0.0

    peak_vram_mb = max(train_peak_vram_mb, test_peak_vram_mb)

    trainer.save_model(os.path.join(model_output_dir, "best_model"))
    tokenizer.save_pretrained(os.path.join(model_output_dir, "best_model"))

    train_runtime = train_result.metrics.get("train_runtime", 0.0)
    train_samples_per_sec = train_result.metrics.get(
        "train_samples_per_second", 0.0
    )

    inference_runtime = test_results.get(
        "test_runtime",
        measured_inference_runtime,
    )
    inference_samples_per_sec = test_results.get(
        "test_samples_per_second",
        len(test_model_dataset) / max(inference_runtime, 1e-9),
    )

    total_experiment_time = time.time() - experiment_start

    print(
        f"{model_name} selesai | "
        f"Test EM: {test_results['test_exact_match']:.2f} | "
        f"Test F1: {test_results['test_f1']:.2f}"
    )

    results = {
        "model_name": model_name,
        "test_loss": test_results.get("test_loss", np.nan),
        "exact_match": test_results["test_exact_match"],
        "f1": test_results["test_f1"],
        "train_runtime_sec": round(train_runtime, 2),
        "train_samples_per_sec": round(train_samples_per_sec, 2),
        "inference_runtime_sec": round(inference_runtime, 2),
        "inference_samples_per_sec": round(inference_samples_per_sec, 2),
        "peak_vram_mb": round(peak_vram_mb, 2),
        "total_experiment_time_sec": round(total_experiment_time, 2),
    }

    del trainer, model, tokenizer
    del train_features, validation_features, test_features
    del train_model_dataset, validation_model_dataset, test_model_dataset
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results

# -----------------------------
# 7. JALANKAN PERBANDINGAN
# -----------------------------

experiment_results = []

for config in MODEL_CONFIGS:
    result = train_and_evaluate_qa(config, dataset_dict)
    experiment_results.append(result)

results_df = pd.DataFrame(experiment_results)

print("\n--- FINAL QA COMPARISON ---")
print(results_df.to_string(index=False))

os.makedirs(OUTPUT_DIR, exist_ok=True)
results_df.to_csv(
    os.path.join(OUTPUT_DIR, "qa_comparison_results.csv"),
    index=False,
)

print("\nEksperimen selesai. Hasil disimpan di:")
print(os.path.join(OUTPUT_DIR, "qa_comparison_results.csv"))

Original samples       : 797
Valid samples          : 792
Corrected offsets      : 409
Corrected capitalization: 1
Unresolved samples     : 5

Samples requiring manual inspection:
(277, 'kesehatan_001', 'Answer not found or ambiguous in context')
(327, 'kompasiana_008', 'Answer not found or ambiguous in context')
(475, 'sekolah_001', 'Answer not found or ambiguous in context')
(477, 'sekolah_001', 'Answer not found or ambiguous in context')
(523, 'kompasiana_014', 'Answer not found or ambiguous in context')

--- DATASET SPLIT SUMMARY ---
train: 554 samples | 32 articles
validation: 120 samples | 7 articles
test: 118 samples | 7 articles
Article overlap: None
Selected split seed: 158

Memulai eksperimen: BERT-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenisasi train...


... (38 outputs left out)

Evaluasi final pada test set: ModernBERT-base...


Training Loss,Validation Loss,Epoch,Exact Match,F1
No log,1.937045,3,15.254237,51.701379


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ModernBERT-base selesai | Test EM: 15.25 | Test F1: 51.70

--- FINAL QA COMPARISON ---
       model_name  test_loss  exact_match        f1  train_runtime_sec  train_samples_per_sec  inference_runtime_sec  inference_samples_per_sec  peak_vram_mb  total_experiment_time_sec
BERT-base-uncased   1.859186    22.033898 57.807354             143.63                  11.57                   0.82                     144.48       2688.62                     159.64
  ModernBERT-base   1.937045    15.254237 51.701379             292.69                   5.68                   1.58                      74.46       3917.55                     400.67

Eksperimen selesai. Hasil disimpan di:
./results/qa_comparison_results.csv


In [5]:
tokenizer_for_length = AutoTokenizer.from_pretrained("bert-base-uncased")

def get_token_length(example):
    return {"context_token_length": len(tokenizer_for_length(example["context"])["input_ids"])}

# Apply the function to the dataset
dataset_with_lengths = hf_dataset.map(get_token_length, num_proc=1) # num_proc=1 to avoid issues if not running in a multiprocessing environment

# Calculate the average token length
average_context_token_length = np.mean(dataset_with_lengths["context_token_length"])

print(f"Rata-rata panjang token pada setiap context: {average_context_token_length:.2f}")

Map (num_proc=1):   0%|          | 0/792 [00:00<?, ? examples/s]

Rata-rata panjang token pada setiap context: 51.78


In [6]:
print(results_df)

          model_name  test_loss  exact_match         f1  train_runtime_sec  \
0  BERT-base-uncased   1.859186    22.033898  57.807354             143.63   
1    ModernBERT-base   1.937045    15.254237  51.701379             292.69   

   train_samples_per_sec  inference_runtime_sec  inference_samples_per_sec  \
0                  11.57                   0.82                     144.48   
1                   5.68                   1.58                      74.46   

   peak_vram_mb  total_experiment_time_sec  
0       2688.62                     159.64  
1       3917.55                     400.67  


In [7]:
print("=== PARAMETER & HYPERPARAMETER YANG DIGUNAKAN ===")
print(f"Seed                  : {SEED}")
print(f"Max Length            : {MAX_LENGTH}")
print(f"Doc Stride            : {DOC_STRIDE}")
print(f"N Best (Predictions)  : {N_BEST}")
print(f"Max Answer Length     : {MAX_ANSWER_LENGTH}")
print(f"Batch Size            : {BATCH_SIZE}")
print(f"Learning Rate         : {LEARNING_RATE}")
print(f"Epochs                : {EPOCHS}")
print(f"Warmup Ratio          : {WARMUP_RATIO}")
print(f"Weight Decay          : {WEIGHT_DECAY}")
print(f"Output Directory      : {OUTPUT_DIR}")
print("\n=== MODEL YANG DIBANDINGKAN ===")
for i, cfg in enumerate(MODEL_CONFIGS, 1):
    print(f"{i}. {cfg['model_name']} ({cfg['checkpoint']})")

=== PARAMETER & HYPERPARAMETER YANG DIGUNAKAN ===
Seed                  : 42
Max Length            : 384
Doc Stride            : 128
N Best (Predictions)  : 20
Max Answer Length     : 30
Batch Size            : 8
Learning Rate         : 3e-05
Epochs                : 3
Warmup Ratio          : 0.1
Weight Decay          : 0.01
Output Directory      : ./results

=== MODEL YANG DIBANDINGKAN ===
1. BERT-base-uncased (bert-base-uncased)
2. ModernBERT-base (answerdotai/ModernBERT-base)


In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# ANALISIS 1: PERBANDINGAN EM vs F1
# ==========================================
print("=== 1. PERBANDINGAN EXACT MATCH (EM) DAN F1 ===")
print(results_df[['model_name', 'exact_match', 'f1', 'test_loss']])

# ==========================================
# ANALISIS 2: PARTIAL MATCH ANALYSIS
# ==========================================
# Kita definisikan kriteria partial match:
# Jawaban tidak pas sempurna (EM = 0), namun memiliki F1 > 0 (artinya ada overlap kata)

# Mari kita simulasikan kembali prediksi dari evaluasi untuk setiap model guna analisis detail
# (Menggunakan fungsi-fungsi pembantu yang sudah didefinisikan sebelumnya)

def analyze_predictions(model_config, raw_splits):
    checkpoint = model_config["checkpoint"]
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)

    # Ambil dataset evaluasi (test)
    test_set = raw_splits["test"]
    test_features = test_set.map(
        lambda batch: prepare_qa_features(batch, tokenizer),
        batched=True,
        remove_columns=test_set.column_names,
    )

    # Setup metadata
    test_metadata = [
        {
            "example_id": int(test_features[i]["example_id"]),
            "offset_mapping": test_features[i]["offset_mapping"],
        }
        for i in range(len(test_features))
    ]

    # Load best model
    model_path = os.path.join(OUTPUT_DIR, model_config["output_name"], "best_model")
    model = AutoModelForQuestionAnswering.from_pretrained(model_path)

    # Predict
    trainer = Trainer(
        model=model,
        data_collator=DefaultDataCollator()
    )

    test_model_dataset = test_features.remove_columns(["example_id", "offset_mapping"])
    predictions = trainer.predict(test_model_dataset)
    start_logits, end_logits = predictions.predictions

    best_predictions = {}
    for feature_index, feature in enumerate(test_metadata):
        example_id = int(feature["example_id"])
        offsets = feature["offset_mapping"]
        context = test_set[example_id]["context"]

        start_indexes = np.argsort(start_logits[feature_index])[-N_BEST:][::-1]
        end_indexes = np.argsort(end_logits[feature_index])[-N_BEST:][::-1]

        for start_index in start_indexes:
            if start_index >= len(offsets) or offsets[start_index] is None: continue
            for end_index in end_indexes:
                if end_index < start_index or end_index >= len(offsets): continue
                if offsets[end_index] is None: continue
                if end_index - start_index + 1 > MAX_ANSWER_LENGTH: continue

                start_char = offsets[start_index][0]
                end_char = offsets[end_index][1]
                if end_char <= start_char: continue

                prediction_text = context[start_char:end_char]
                score = float(start_logits[feature_index][start_index]) + float(end_logits[feature_index][end_index])

                current = best_predictions.get(example_id)
                if current is None or score > current["score"]:
                    best_predictions[example_id] = {
                        "text": prediction_text,
                        "score": score,
                    }

    # Hitung metrik detail per sampel
    detailed_records = []
    for idx, example in enumerate(test_set):
        pred = best_predictions.get(idx, {}).get("text", "")
        gold = example["answers"]["text"][0] if example["answers"]["text"] else ""

        em_score = float(normalize_answer(pred) == normalize_answer(gold))
        f1_score = calculate_text_f1(pred, gold)

        status = "Incorrect"
        if em_score == 1.0:
            status = "Exact Match"
        elif f1_score > 0.0:
            status = "Partial Match"

        detailed_records.append({
            "question": example["question"],
            "gold": gold,
            "pred": pred,
            "em": em_score,
            "f1": f1_score,
            "status": status,
            "pred_len": len(normalize_answer(pred).split()),
            "gold_len": len(normalize_answer(gold).split())
        })

    return pd.DataFrame(detailed_records)

# Jalankan analisis untuk kedua model
model_analyses = {}
for config in MODEL_CONFIGS:
    name = config["model_name"]
    print(f"Analyzing predictions for {name}...")
    model_analyses[name] = analyze_predictions(config, dataset_dict)

print("\n=== 2. PARTIAL MATCH ANALYSIS ===")
for name, df_analysis in model_analyses.items():
    status_counts = df_analysis["status"].value_counts()
    print(f"\nDistribusi Hasil untuk {name}:")
    for status, count in status_counts.items():
        pct = (count / len(df_analysis)) * 100
        print(f" - {status}: {count} ({pct:.2f}%)")

# ==========================================
# ANALISIS 3: ANSWER LENGTH ANALYSIS
# ==========================================
print("\n=== 3. ANSWER LENGTH ANALYSIS (Karakteristik Panjang Jawaban) ===")
for name, df_analysis in model_analyses.items():
    avg_gold_len = df_analysis["gold_len"].mean()
    avg_pred_len = df_analysis["pred_len"].mean()
    correlation = df_analysis["gold_len"].corr(df_analysis["pred_len"])

    print(f"\nModel: {name}")
    print(f" - Rata-rata panjang jawaban asli (words) : {avg_gold_len:.2f}")
    print(f" - Rata-rata panjang jawaban prediksi (words): {avg_pred_len:.2f}")
    print(f" - Korelasi panjang asli vs prediksi     : {correlation:.4f}")

=== 1. PERBANDINGAN EXACT MATCH (EM) DAN F1 ===
          model_name  exact_match         f1  test_loss
0  BERT-base-uncased    22.033898  57.807354   1.859186
1    ModernBERT-base    15.254237  51.701379   1.937045
Analyzing predictions for BERT-base-uncased...


Map:   0%|          | 0/118 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Analyzing predictions for ModernBERT-base...


Map:   0%|          | 0/118 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]


=== 2. PARTIAL MATCH ANALYSIS ===

Distribusi Hasil untuk BERT-base-uncased:
 - Partial Match: 73 (61.86%)
 - Exact Match: 26 (22.03%)
 - Incorrect: 19 (16.10%)

Distribusi Hasil untuk ModernBERT-base:
 - Partial Match: 81 (68.64%)
 - Incorrect: 19 (16.10%)
 - Exact Match: 18 (15.25%)

=== 3. ANSWER LENGTH ANALYSIS (Karakteristik Panjang Jawaban) ===

Model: BERT-base-uncased
 - Rata-rata panjang jawaban asli (words) : 5.31
 - Rata-rata panjang jawaban prediksi (words): 6.19
 - Korelasi panjang asli vs prediksi     : 0.2655

Model: ModernBERT-base
 - Rata-rata panjang jawaban asli (words) : 5.31
 - Rata-rata panjang jawaban prediksi (words): 6.74
 - Korelasi panjang asli vs prediksi     : 0.2081
